# Sum of Square Spectral Amplification Resource Estimation

This notebook demonstrates how to use QDK Chemistry to construct a Sum of Squares Spectral Amplification (SOSSA) block-encoding circuit and perform resource estimation using QRE v3.
The goal is to enable users to understand and further improve on the fault-tolerant resource requirements of state-of-the-art quantum algorithms.

In addition to [installing `qdk-chemistry`](https://github.com/microsoft/qdk-chemistry/blob/main/INSTALL.md), you will need to install the `jupyter` and `qre` extras to run this notebook:

```bash
pip install 'qdk-chemistry[jupyter,qre]'
```

In [1]:
# Reduce logging output for demo
from qdk_chemistry.utils import Logger
Logger.set_global_level(Logger.LogLevel.off)

## Setup a factorized Hamiltonian for H2

**No DFTHC factorization required.** This notebook uses only `(N, R, B, C, b_rot, b_coeff, lambda_eff)` parameters to generate fake data of correct size.

In [2]:
from math import sqrt

import numpy as np
import pandas as pd

from qdk_chemistry.algorithms.controlled_circuit_mapper.sossa_mapper import (
    InnerPrepareMapper,
    OuterPrepareMapper,
    SelectMapper,
    SOSSAMapper,
)
from qdk_chemistry.algorithms.hamiltonian_unitary_builder.block_encoding.sossa import SOSSABuilder
from qdk_chemistry.data import BasisSet, FactorizedHamiltonianContainer, Orbitals, OrbitalType, Shell
from qdk_chemistry.data.controlled_unitary import ControlledUnitary
from qdk_chemistry.data.unitary_representation.containers.sossa import SOSSAContainer

# H2-like DFTHC factorized Hamiltonian: N=2 orbitals, R=1, B=1, C=1
N, R, B, C = 2, 1, 1, 1

h1 = np.array([
    [0.3, 0.1],
    [0.1, -0.2],
])
basis_vectors = np.array([[[1.0 / sqrt(2), 1.0 / sqrt(2)]]])  # [R, B, N]
two_body_weights = np.array([[[0.15]]])                         # [R, B, C]
identity_weight = np.array([[0.08]])                            # [R, C]

# Build FactorizedHamiltonianContainer
shells = [Shell(0, OrbitalType.S, np.array([1.0]), np.array([1.0])) for _ in range(N)]
basis_set = BasisSet("test-basis", shells)
orbitals = Orbitals(np.eye(N), None, None, basis_set)
inactive_fock = np.zeros((N, N))

fh = FactorizedHamiltonianContainer(
    h1, basis_vectors.flatten(),
    two_body_weights.flatten(), identity_weight,
    R, B, C,
    orbitals, 0.0, inactive_fock,
)
print(f"Factorized Hamiltonian: N={N}, R={R}, B={B}, C={C}")

Factorized Hamiltonian: N=2, R=1, B=1, C=1


## Generate SOSSA Circuit

In [3]:
# Step 1: SOSSABuilder → UnitaryRepresentation
builder = SOSSABuilder(quantum_walk=True)
unitary_rep = builder.run(fh)
container = unitary_rep.get_container()

# Step 2: SOSSAMapper → Circuit
controlled_unitary = ControlledUnitary(unitary=unitary_rep, control_indices=[0])
mapper = SOSSAMapper(
    outer_prepare_mapper=OuterPrepareMapper(algorithm="dense_pure"),
    inner_prepare_mapper=InnerPrepareMapper(algorithm="direct"),
    select_mapper=SelectMapper(multiplexed_rotation="direct"),
)
circuit = mapper.run(controlled_unitary)
lambda_sos = container.normalization
print(f"SOSSA normalization Λ = {lambda_sos:.6f}")
print(f"Circuit built successfully")

SOSSA normalization Λ = 1.122596
Circuit built successfully


## Verify QPE simulation

In [4]:
from qdk_chemistry.algorithms.phase_estimation.iterative_phase_estimation import IterativePhaseEstimation
from qdk_chemistry.data import AlgorithmRef
from qdk_chemistry.data.circuit import Circuit, QsharpFactoryData
from qdk_chemistry.utils.qsharp import QSHARP_UTILS

# Hartree-Fock state |1100⟩ as trial state (2 electrons in lowest orbitals)
num_system_qubits = 2 * N
hf_bitstring = [1, 1] + [0] * (num_system_qubits - 2)
state_prep_params = {
    "rowMap": list(range(num_system_qubits - 1, -1, -1)),
    "stateVector": [0.0] * (2**num_system_qubits),
    "expansionOps": [],
    "numQubits": num_system_qubits,
}
# Set HF amplitude: |1100⟩ in big-endian = index 12
hf_index = sum(b << (num_system_qubits - 1 - i) for i, b in enumerate(hf_bitstring))
state_prep_params["stateVector"][hf_index] = 1.0

qsharp_factory = QsharpFactoryData(
    program=QSHARP_UTILS.StatePreparation.MakeStatePreparationCircuit,
    parameter=state_prep_params,
)
qsharp_op = QSHARP_UTILS.StatePreparation.MakeStatePreparationOp(state_prep_params)
state_prep = Circuit(qsharp_factory=qsharp_factory, qsharp_op=qsharp_op)

# Run IQPE with SOSSA walk operator
num_bits = 5
iqpe = IterativePhaseEstimation(shots_per_bit=5)
iqpe.settings().set("qpe_circuit_builder", AlgorithmRef(
    "qpe_circuit_builder", "qdk_iterative",
    num_bits=num_bits,
    controlled_circuit_mapper=AlgorithmRef(
        "controlled_circuit_mapper", "sossa",
        outer_prepare_algorithm="dense_pure",
        inner_prepare_algorithm="direct",
        select_algorithm="direct",
    ),
    unitary_builder=AlgorithmRef("hamiltonian_unitary_builder", "sossa", quantum_walk=True),
))
iqpe.settings().set(
    "circuit_executor",
    AlgorithmRef("circuit_executor", "qdk_sparse_state_simulator"),
)

result = iqpe.run(
    state_preparation=state_prep,
    factorized_hamiltonian=fh,
)
print(f"QPE phase: {result.phase_fraction:.6f}")
print(f"QPE raw energy (Λ·cos 2πφ): {result.raw_energy:.6f}")
print(f"Energy gap estimate: {lambda_sos - result.raw_energy:.6f}")

QPE phase: 0.000000
QPE raw energy (Λ·cos 2πφ): 1.122596
Energy gap estimate: 0.000000


## Standard QPE Simulation

Use the standard (QFT-based) QPE algorithm with the SOSSA walk operator to estimate the ground-state energy.

In [5]:
from qdk_chemistry.algorithms.phase_estimation.standard_phase_estimation import StandardPhaseEstimation

# Run standard (QFT-based) QPE with SOSSA walk operator
num_bits_std = 5
std_qpe = StandardPhaseEstimation(shots=3)
std_qpe.settings().set("qpe_circuit_builder", AlgorithmRef(
    "qpe_circuit_builder", "qdk_standard",
    num_bits=num_bits_std,
    controlled_circuit_mapper=AlgorithmRef(
        "controlled_circuit_mapper", "sossa",
        outer_prepare_algorithm="dense_pure",
        inner_prepare_algorithm="direct",
        select_algorithm="direct",
    ),
    unitary_builder=AlgorithmRef("hamiltonian_unitary_builder", "sossa", quantum_walk=True),
))
std_qpe.settings().set(
    "circuit_executor",
    AlgorithmRef("circuit_executor", "qdk_sparse_state_simulator"),
)

std_result = std_qpe.run(
    state_preparation=state_prep,
    factorized_hamiltonian=fh,
)
print(f"Standard QPE phase: {std_result.phase_fraction:.6f}")
print(f"Standard QPE raw energy (Λ·cos 2πφ): {std_result.raw_energy:.6f}")
print(f"Energy gap estimate: {lambda_sos - std_result.raw_energy:.6f}")

Standard QPE phase: 0.000000
Standard QPE raw energy (Λ·cos 2πφ): 1.122596
Energy gap estimate: 0.000000


## QRE v3 Resource Estimation (Standard QPE)

Build the full standard QPE circuit and use `circuit.estimate()` (QRE v3) for logical resource counts.

In [6]:
from qdk_chemistry.algorithms.phase_estimation.circuit_builder.standard_builder import QdkStandardQpeCircuitBuilder

# Build the full standard QPE circuit for resource estimation
num_bits_re_std = 10

std_builder = QdkStandardQpeCircuitBuilder(
    num_bits=num_bits_re_std,
    controlled_circuit_mapper=AlgorithmRef(
        "controlled_circuit_mapper", "sossa",
        outer_prepare_algorithm="dense_pure",
        inner_prepare_algorithm="direct",
        select_algorithm="direct",
    ),
    unitary_builder=AlgorithmRef("hamiltonian_unitary_builder", "sossa", quantum_walk=True),
)

std_circuits = std_builder.run(
    state_preparation=state_prep,
    factorized_hamiltonian=fh,
)
std_qpe_circuit = std_circuits[0]

# QRE v3 resource estimation via qsharp.estimate()
std_qpe_estimate = std_qpe_circuit.estimate()
df_std_qpe = pd.DataFrame(
    std_qpe_estimate.logical_counts.items(),
    columns=["Logical Estimate", "Counts"],
)
print(f"Standard QPE logical resource counts ({num_bits_re_std}-bit precision):")
display(df_std_qpe)

Standard QPE logical resource counts (10-bit precision):


,Logical Estimate,Counts
0,numQubits,27
1,tCount,8211
2,rotationCount,30798
3,rotationDepth,30746
4,cczCount,136059
5,ccixCount,0
6,measurementCount,15355


In [7]:
# QRE v3 via qdk.qre.estimate() (advanced path)
try:
    import qdk.qre as qre

    app = std_qpe_circuit.get_qre_application()
    qre_result = qre.estimate(app)
    print("QRE v3 (qdk.qre) estimation complete:")
    print(f"  Physical qubits: {qre_result.physical_counts.get('physicalQubits', 'N/A')}")
    print(f"  Runtime: {qre_result.physical_counts.get('runtime', 'N/A')}")
except (ImportError, RuntimeError) as e:
    print(f"qdk.qre not available (expected in dev): {e}")
    print("Falling back to qsharp.estimate() results above.")

TypeError: estimate() missing 2 required positional arguments: 'architecture' and 'isa_query'